# Hospital / Emergency Services Data

## Loading Hospital Data: Filtering to NY/NJ/CT

In [1]:
import pandas as pd
import requests

hospitals = pd.read_csv(
    "Hospital_General_Information.csv"
)
ct_towns = pd.read_csv("ct_town_crosswalk.csv")

hospitals = hospitals[
    hospitals["State"].isin(["NY", "NJ", "CT"])
]

## Getting FIPS Code

In [2]:
fips_df = pd.read_csv("ny_nj_ct_fips.csv")

In [3]:
hospitals.columns = hospitals.columns.str.lower()
fips_df.columns = fips_df.columns.str.lower()

In [4]:
hospitals["county"] = hospitals["county/parish"].str.upper().str.strip()
hospitals["state"] = hospitals["state"].str.upper().str.strip()

fips_df["county"] = fips_df["county"].str.upper().str.strip()
fips_df["state"] = fips_df["state"].str.upper().str.strip()

In [5]:
fips_df["county"] = (
    fips_df["county"]
    .str.upper()
    .str.replace(" COUNTY", "", regex=False)
    .str.strip()
)

In [6]:
fips_df["fips"] = fips_df["fips"].astype(str).str.zfill(5)

In [7]:
hospitals["state"] = hospitals["state"].str.upper().str.strip()
fips_df["state"] = fips_df["state"].str.upper().str.strip()

### Merging Hospitals Dataset with FIPS Dataframe -> Creating FIPS Column

In [8]:
hospitals_geo = hospitals.merge(
    fips_df,
    on=["state", "county"],
    how="left"
)

### Getting Hospital Count Number By County

In [9]:
hospital_counts = hospitals_geo.groupby("fips").size().reset_index(name="hospital_count")

In [10]:
hospital_counts

,fips,hospital_count
0,09001,8
1,09003,8
2,09005,2
3,09007,3
4,09009,9
...,...,...
80,36113,1
81,36117,1
82,36119,12
83,36121,1


### Finding Connecticut County Population and Calculating Hospitals/100k

In [11]:
ct_towns

,town_name,town_fips_2020,county_fips,county_name,town_fips_2022,region_fips,region_name
0,Andover,901301080,9013,Tolland County,911001080,9110,Capitol Planning Region
1,Ansonia,900901220,9009,New Haven County,914001220,9140,Naugatuck Valley Planning Region
2,Ashford,901501430,9015,Windham County,915001430,9150,Northeastern Connecticut Planning Region
3,Avon,900302060,9003,Hartford County,911002060,9110,Capitol Planning Region
4,Barkhamsted,900502760,9005,Litchfield County,916002760,9160,Northwest Hills Planning Region
...,...,...,...,...,...,...,...
164,Windsor Locks,900387070,9003,Hartford County,911087070,9110,Capitol Planning Region
165,Wolcott,900987560,9009,New Haven County,914087560,9140,Naugatuck Valley Planning Region
166,Woodbridge,900987700,9009,New Haven County,917087700,9170,South Central Connecticut Planning Region
167,Woodbury,900587910,9005,Litchfield County,914087910,9140,Naugatuck Valley Planning Region


In [12]:
# Load the population data
pop_df = pd.read_csv('ct_towns_pop2023.csv')

# Standardize town names for merging (uppercase to match ct_towns)
pop_df['town_name'] = pop_df['town_name'].str.upper()

# Merge with ct_towns to get county info
merged = ct_towns[['town_name', 'county_name']].merge(
    pop_df,
    on='town_name',
    how='left'
)

# Group by county and sum populations
county_pop = merged.groupby('county_name')['pop_2023'].sum().reset_index()
county_pop.columns = ['county_name', 'total_pop_2023']
county_pop = county_pop.sort_values('total_pop_2023', ascending=False)

print(county_pop)

         county_name  total_pop_2023
0   Fairfield County             0.0
1    Hartford County             0.0
2  Litchfield County             0.0
3   Middlesex County             0.0
4   New Haven County             0.0
5  New London County             0.0
6     Tolland County             0.0
7     Windham County             0.0


In [13]:
# Check for any towns that didn't match
unmatched = merged[merged['pop_2023'].isna()]['town_name'].tolist()
print("Unmatched towns:", unmatched)

Unmatched towns: ['Andover', 'Ansonia', 'Ashford', 'Avon', 'Barkhamsted', 'Beacon Falls', 'Berlin', 'Bethany', 'Bethel', 'Bethlehem', 'Bloomfield', 'Bolton', 'Bozrah', 'Branford', 'Bridgeport', 'Bridgewater', 'Bristol', 'Brookfield', 'Brooklyn', 'Burlington', 'Canaan', 'Canterbury', 'Canton', 'Chaplin', 'Cheshire', 'Chester', 'Clinton', 'Colchester', 'Colebrook', 'Columbia', 'Cornwall', 'Coventry', 'Cromwell', 'Danbury', 'Darien', 'Deep River', 'Derby', 'Durham', 'East Granby', 'East Haddam', 'East Hampton', 'East Hartford', 'East Haven', 'East Lyme', 'East Windsor', 'Eastford', 'Easton', 'Ellington', 'Enfield', 'Essex', 'Fairfield', 'Farmington', 'Franklin', 'Glastonbury', 'Goshen', 'Granby', 'Greenwich', 'Griswold', 'Groton', 'Guilford', 'Haddam', 'Hamden', 'Hampton', 'Hartford', 'Hartland', 'Harwinton', 'Hebron', 'Kent', 'Killingly', 'Killingworth', 'Lebanon', 'Ledyard', 'Lisbon', 'Litchfield', 'Lyme', 'Madison', 'Manchester', 'Mansfield', 'Marlborough', 'Meriden', 'Middlebury', 'Mi

In [14]:
# Normalize both to uppercase for matching
ct_towns_copy = ct_towns.copy()
ct_towns_copy['town_name_upper'] = ct_towns_copy['town_name'].str.upper()
pop_df['town_name_upper'] = pop_df['town_name'].str.upper()

# Merge on the normalized column
merged = ct_towns_copy[['town_name', 'town_name_upper', 'county_name']].merge(
    pop_df[['town_name_upper', 'pop_2023']],
    on='town_name_upper',
    how='left'
)

# Check unmatched
unmatched = merged[merged['pop_2023'].isna()]['town_name'].tolist()
print("Unmatched towns:", unmatched)

# Group by county
county_pop = merged.groupby('county_name')['pop_2023'].sum().reset_index()
county_pop.columns = ['county_name', 'total_pop_2023']
county_pop = county_pop.sort_values('total_pop_2023', ascending=False)
print(county_pop)

Unmatched towns: []
         county_name  total_pop_2023
0   Fairfield County          963780
1    Hartford County          898478
4   New Haven County          865717
5  New London County          268518
2  Litchfield County          186551
3   Middlesex County          166110
6     Tolland County          150906
7     Windham County          117116


In [15]:
# Get unique county_fips + county_name from ct_towns
county_fips_map = ct_towns[['county_fips', 'county_name']].drop_duplicates()

# Merge fips into county population df
ct_county_pop = county_pop.merge(
    county_fips_map,
    on='county_name',
    how='left'
)
ct_county_pop.set_index('county_fips', inplace = True)
print(ct_county_pop)

                   county_name  total_pop_2023
county_fips                                   
9001          Fairfield County          963780
9003           Hartford County          898478
9009          New Haven County          865717
9011         New London County          268518
9005         Litchfield County          186551
9007          Middlesex County          166110
9013            Tolland County          150906
9015            Windham County          117116


In [16]:
hospitals_per_100k_all = pd.read_csv('hospitals_per_100k_nj_ny.csv')

In [17]:
hospitals_per_100k_all = hospitals_per_100k_all.merge(
    ct_county_pop[['county_name', 'total_pop_2023']],
    left_on='fips',
    right_index=True,
    how='left'
)

hospitals_per_100k_all['county'] = hospitals_per_100k_all['county'].fillna(hospitals_per_100k_all['county_name'])
hospitals_per_100k_all['population'] = hospitals_per_100k_all['population'].fillna(hospitals_per_100k_all['total_pop_2023'])
hospitals_per_100k_all['hospitals_per_100k'] = hospitals_per_100k_all['hospital_count'] / hospitals_per_100k_all['population'] * 100000

hospitals_per_100k_all.drop(columns=['county_name', 'total_pop_2023'], inplace=True)
print(hospitals_per_100k_all[hospitals_per_100k_all['fips'] < 10000])

   fips  hospital_count state             county  population  \
0  9001               8   NaN   Fairfield County    963780.0   
1  9003               8   NaN    Hartford County    898478.0   
2  9005               2   NaN  Litchfield County    186551.0   
3  9007               3   NaN   Middlesex County    166110.0   
4  9009               9   NaN   New Haven County    865717.0   
5  9011               2   NaN  New London County    268518.0   
6  9013               2   NaN     Tolland County    150906.0   
7  9015               2   NaN     Windham County    117116.0   

   hospitals_per_100k  
0            0.830065  
1            0.890395  
2            1.072093  
3            1.806032  
4            1.039601  
5            0.744829  
6            1.325328  
7            1.707709  


In [18]:
# Fill NaN state with Connecticut
hospitals_per_100k_all['state'] = hospitals_per_100k_all['state'].fillna('Connecticut')

# Remove ' County' from CT rows only
hospitals_per_100k_all['county'] = hospitals_per_100k_all['county'].str.replace(' County', '', regex=False)



In [19]:
print(hospitals_per_100k_all.tail())

     fips  hospital_count     state       county  population  \
80  36113               1  New York       Warren     65467.0   
81  36117               1  New York        Wayne     90721.0   
82  36119              12  New York  Westchester    998335.0   
83  36121               1  New York      Wyoming     39750.0   
84  36123               1  New York        Yates     24367.0   

    hospitals_per_100k  
80            1.527487  
81            1.102281  
82            1.202001  
83            2.515723  
84            4.103911  


In [20]:
#hospitals_per_100k_all.to_csv('hospitals_per_100k_all.csv', index=False)

### PCP per 100k

In [21]:
#Get Primary Care Physican data from County Health Rankings
df_raw = pd.read_excel('County_Health_Rankings_NY.xlsx', 
                        sheet_name='Ranked Measure Data', header=None)

cols = [0, 1, 2, 114]
pcp_df = df_raw.iloc[2:, cols].copy()

pcp_df.columns = ['FIPS', 'State', 'County', 
                   '# Primary Care Physicians']

pcp_df = pcp_df.iloc[1:]
pcp_df = pcp_df.reset_index(drop=True)

In [22]:
pcp_df.head()

,FIPS,State,County,# Primary Care Physicians
0,36001,New York,Albany,307
1,36003,New York,Allegany,19
2,36005,New York,Bronx,907
3,36007,New York,Broome,163
4,36009,New York,Cattaraugus,35


In [23]:
df_raw = pd.read_excel('County_Health_Rankings_CT.xlsx', 
                        sheet_name='Ranked Measure Data', header=None)

cols = [0, 1, 2, 114]
pcp_df2= df_raw.iloc[2:, cols].copy()

pcp_df2.columns = ['FIPS', 'State', 'County', 
                   '# Primary Care Physicians']

pcp_df2 = pcp_df2.iloc[1:]
pcp_df2 = pcp_df2.reset_index(drop=True)

In [24]:
pcp_df2.head()

,FIPS,State,County,# Primary Care Physicians
0,09001,Connecticut,Fairfield,918
1,09003,Connecticut,Hartford,856
2,09005,Connecticut,Litchfield,100
3,09007,Connecticut,Middlesex,129
4,09009,Connecticut,New Haven,740


In [25]:
df_raw = pd.read_excel('County_Health_Rankings_NJ.xlsx', 
                        sheet_name='Ranked Measure Data', header=None)

cols = [0, 1, 2, 114]
pcp_df3= df_raw.iloc[2:, cols].copy()

pcp_df3.columns = ['FIPS', 'State', 'County', 
                   '# Primary Care Physicians']
pcp_df3 = pcp_df3.iloc[1:]
pcp_df3 = pcp_df3.reset_index(drop=True)

In [26]:
pcp_df3.head()

,FIPS,State,County,# Primary Care Physicians
0,34001,New Jersey,Atlantic,211
1,34003,New Jersey,Bergen,1110
2,34005,New Jersey,Burlington,377
3,34007,New Jersey,Camden,509
4,34009,New Jersey,Cape May,51


In [27]:
pcp_all = pd.concat([pcp_df2, pcp_df3, pcp_df], join = 'inner')
pcp_all = pcp_all.rename(columns={'FIPS' :'fips'})
pcp_all.set_index('fips', inplace = True)
pcp_all.head(20)

,State,County,# Primary Care Physicians
fips,,,
09001,Connecticut,Fairfield,918
09003,Connecticut,Hartford,856
09005,Connecticut,Litchfield,100
09007,Connecticut,Middlesex,129
09009,Connecticut,New Haven,740
09011,Connecticut,New London,167
09013,Connecticut,Tolland,81
09015,Connecticut,Windham,52
34001,New Jersey,Atlantic,211


In [28]:
import requests

url = "https://api.census.gov/data/2023/acs/acs1"

API_KEY = "3d1a8efb010ab94526aed9bb9b1b8ba58c722e3d"

states = ["09", "34", "36"]  # CT, NJ, NY

all_data = []

for s in states:
    params = {
        "get": "NAME,B01003_001E",
        "for": "county:*",
        "in": f"state:{s}",
        "key": API_KEY
    }

    r = requests.get(url, params=params)

    print("\nSTATE:", s)
    print("STATUS:", r.status_code)
    print("CONTENT (first 300 chars):")
    print(r.text[:300])
    print("-" * 50)


STATE: 09
STATUS: 200
CONTENT (first 300 chars):
[["NAME","B01003_001E","state","county"],
["Capitol Planning Region, Connecticut","975328","09","110"],
["Greater Bridgeport Planning Region, Connecticut","327651","09","120"],
["Lower Connecticut River Valley Planning Region, Connecticut","176215","09","130"],
["Naugatuck Valley Planning Region, Co
--------------------------------------------------

STATE: 34
STATUS: 200
CONTENT (first 300 chars):
[["NAME","B01003_001E","state","county"],
["Atlantic County, New Jersey","275213","34","001"],
["Bergen County, New Jersey","957736","34","003"],
["Burlington County, New Jersey","469167","34","005"],
["Camden County, New Jersey","527196","34","007"],
["Cape May County, New Jersey","94610","34","009
--------------------------------------------------

STATE: 36
STATUS: 200
CONTENT (first 300 chars):
[["NAME","B01003_001E","state","county"],
["Albany County, New York","316659","36","001"],
["Bronx County, New York","1356476","36","005"],
["Broo

In [29]:
pop = pd.read_csv("co-est2025-alldata.csv", encoding = "latin1")

In [30]:
print(pop.columns.tolist())

['SUMLEV', 'REGION', 'DIVISION', 'STATE', 'COUNTY', 'STNAME', 'CTYNAME', 'ESTIMATESBASE2020', 'POPESTIMATE2020', 'POPESTIMATE2021', 'POPESTIMATE2022', 'POPESTIMATE2023', 'POPESTIMATE2024', 'POPESTIMATE2025', 'NPOPCHG2020', 'NPOPCHG2021', 'NPOPCHG2022', 'NPOPCHG2023', 'NPOPCHG2024', 'NPOPCHG2025', 'BIRTHS2020', 'BIRTHS2021', 'BIRTHS2022', 'BIRTHS2023', 'BIRTHS2024', 'BIRTHS2025', 'DEATHS2020', 'DEATHS2021', 'DEATHS2022', 'DEATHS2023', 'DEATHS2024', 'DEATHS2025', 'NATURALCHG2020', 'NATURALCHG2021', 'NATURALCHG2022', 'NATURALCHG2023', 'NATURALCHG2024', 'NATURALCHG2025', 'INTERNATIONALMIG2020', 'INTERNATIONALMIG2021', 'INTERNATIONALMIG2022', 'INTERNATIONALMIG2023', 'INTERNATIONALMIG2024', 'INTERNATIONALMIG2025', 'DOMESTICMIG2020', 'DOMESTICMIG2021', 'DOMESTICMIG2022', 'DOMESTICMIG2023', 'DOMESTICMIG2024', 'DOMESTICMIG2025', 'NETMIG2020', 'NETMIG2021', 'NETMIG2022', 'NETMIG2023', 'NETMIG2024', 'NETMIG2025', 'RESIDUAL2020', 'RESIDUAL2021', 'RESIDUAL2022', 'RESIDUAL2023', 'RESIDUAL2024', 'RES

In [31]:
# Filtering for NY-NJ-CT
pop = pop[
    pop["STATE"].isin([9, 34, 36])
]

In [32]:
pop = pop[
    pop["COUNTY"] > 0
]

In [33]:
pop["fips"] = (
    pop["STATE"].astype(str).str.zfill(2)
    + pop["COUNTY"].astype(str).str.zfill(3)
)

In [34]:
pop = pop[[
    "fips",
    "STNAME",
    "CTYNAME",
    "POPESTIMATE2023"
]]

In [35]:
# Renaming columns
pop = pop.rename(columns={
    "STNAME": "state",
    "CTYNAME": "county",
    "POPESTIMATE2023": "population"
})

In [36]:
# Check data
print(pop.shape)
print(pop.head())

(92, 4)
      fips        state                                          county  \
316  09110  Connecticut                         Capitol Planning Region   
317  09120  Connecticut              Greater Bridgeport Planning Region   
318  09130  Connecticut  Lower Connecticut River Valley Planning Region   
319  09140  Connecticut                Naugatuck Valley Planning Region   
320  09150  Connecticut        Northeastern Connecticut Planning Region   

     population  
316      981775  
317      332081  
318      176419  
319      457384  
320       96790  


In [37]:
pop = pop[pop['state'] != 'Connecticut']

In [38]:
pop.head()

,fips,state,county,population
1807,34001,New Jersey,Atlantic County,276643
1808,34003,New Jersey,Bergen County,964156
1809,34005,New Jersey,Burlington County,472273
1810,34007,New Jersey,Camden County,529977
1811,34009,New Jersey,Cape May County,94577


In [39]:
ct_county_pop.head()

,county_name,total_pop_2023
county_fips,,
9001,Fairfield County,963780
9003,Hartford County,898478
9009,New Haven County,865717
9011,New London County,268518
9005,Litchfield County,186551


In [40]:
ct_county_pop = ct_county_pop.rename(columns={'county_name': 'county', 'total_pop_2023':'population'})
ct_county_pop.index.name = 'fips'
ct_county_pop.head()

,county,population
fips,,
9001,Fairfield County,963780
9003,Hartford County,898478
9009,New Haven County,865717
9011,New London County,268518
9005,Litchfield County,186551


In [41]:
ct_county_pop['state'] = 'Connecticut'
pop['fips'] = pop['fips'].astype(int)
pop.set_index('fips', inplace = True)

In [42]:
pop.head()

,state,county,population
fips,,,
34001,New Jersey,Atlantic County,276643
34003,New Jersey,Bergen County,964156
34005,New Jersey,Burlington County,472273
34007,New Jersey,Camden County,529977
34009,New Jersey,Cape May County,94577


In [43]:
pop_new = pd.concat([ct_county_pop, pop])

In [44]:
pop_new.head(20)

,county,population,state
fips,,,
9001,Fairfield County,963780,Connecticut
9003,Hartford County,898478,Connecticut
9009,New Haven County,865717,Connecticut
9011,New London County,268518,Connecticut
9005,Litchfield County,186551,Connecticut
9007,Middlesex County,166110,Connecticut
9013,Tolland County,150906,Connecticut
9015,Windham County,117116,Connecticut
34001,Atlantic County,276643,New Jersey


In [45]:
pop_new.index = pop_new.index.astype(int)
pcp_all.index =pcp_all.index.astype(int)

merged = pcp_all.merge(pop_new[['population']], left_on='fips', right_index=True, how='left')

merged['pcp_per_100k'] = merged['# Primary Care Physicians'] / merged['population'] * 100000

In [46]:
merged.head(20)

,State,County,# Primary Care Physicians,population,pcp_per_100k
fips,,,,,
9001,Connecticut,Fairfield,918,963780,95.249953
9003,Connecticut,Hartford,856,898478,95.272227
9005,Connecticut,Litchfield,100,186551,53.604644
9007,Connecticut,Middlesex,129,166110,77.659382
9009,Connecticut,New Haven,740,865717,85.47828
9011,Connecticut,New London,167,268518,62.193224
9013,Connecticut,Tolland,81,150906,53.675798
9015,Connecticut,Windham,52,117116,44.400424
34001,New Jersey,Atlantic,211,276643,76.271585


In [47]:
merged[merged['pcp_per_100k'] < 30]

,State,County,# Primary Care Physicians,population,pcp_per_100k
fips,,,,,
34033,New Jersey,Salem,19,65488,29.012949
36011,New York,Cayuga,19,74563,25.481807
36037,New York,Genesee,17,58016,29.302261
36041,New York,Hamilton,1,5068,19.73165
36043,New York,Herkimer,17,59551,28.54696
36073,New York,Orleans,3,39567,7.582076
36095,New York,Schoharie,9,30140,29.86065
36099,New York,Seneca,9,32449,27.735832
36107,New York,Tioga,14,47606,29.408058


In [48]:
#merged.to_csv('pcp_per_100k_all.csv', index=False)

## Stroke Centers Per 100k

#### Importing and Cleaning NY

In [49]:
from bs4 import BeautifulSoup


headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
}

url = "https://www.health.ny.gov/diseases/cardiovascular/stroke/designation/stroke_designated_centers.htm"
response = requests.get(url, headers=headers)
response.raise_for_status()

soup = BeautifulSoup(response.text, "lxml")
table = soup.find("table")

headers_row = [th.get_text(strip=True) for th in table.find_all("th")]

rows = []
for tr in table.find_all("tr")[1:]:  # skip header row
    cells = [td.get_text(strip=True) for td in tr.find_all("td")]
    if len(cells) == len(headers_row):  # skip the "no centers in county" placeholder row
        rows.append(cells)

df = pd.DataFrame(rows, columns=headers_row)
print(df.shape)
print(df.head())

(123, 6)
                                       Hospital Name   PFI   County  \
0  There are no NYSDOH Stroke Designated Centers ...                  
1                              Albany Medical Center  0001   Albany   
2                         Arnot Ogden Medical Center  0116  Chemung   
3                          Auburn Community Hospital  0085   Cayuga   
4                           BronxCare Health Systems  1178    Bronx   

                                 Address    EMS Region  \
0                                                        
1  43 New Scotland Ave, Albany, NY 12208  Northeastern   
2          600 Roe Ave, Elmira, NY 14905  Finger Lakes   
3        17 Lansing St, Auburn, NY 13021       Central   
4  1650 Grand Concourse, Bronx, NY 10457         Bronx   

   NYSDOH Level of Designation  
0                               
1  Comprehensive Stroke Center  
2        Primary Stroke Center  
3        Primary Stroke Center  
4        Primary Stroke Center  


In [50]:
#Cleaning table
df = df.iloc[1:]
df = df.drop(columns=['PFI', 'Address', 'EMS Region'])
df.head(20)

,Hospital Name,County,NYSDOH Level of Designation
1,Albany Medical Center,Albany,Comprehensive Stroke Center
2,Arnot Ogden Medical Center,Chemung,Primary Stroke Center
3,Auburn Community Hospital,Cayuga,Primary Stroke Center
4,BronxCare Health Systems,Bronx,Primary Stroke Center
5,Brookdale Hospital Medical Center,Kings,Comprehensive Stroke Center
6,Brooklyn Hospital Center,Kings,Primary Stroke Center
7,Buffalo General Medical Center,Erie,Comprehensive Stroke Center
8,Canton-Potsdam Hospital,St. Lawrence,Primary Stroke Center
9,Cayuga Medical Center,Tompkins,Primary Stroke Center
10,Columbia Memorial Hospital,Columbia,Primary Stroke Center


In [51]:
#Finding # stroke centers by county 
stroke_centers_by_county = (
    df.groupby("County")["Hospital Name"]
    .count()
    .reset_index()
    .rename(columns={"Hospital Name": "Total Stroke Centers"})
    .sort_values("Total Stroke Centers", ascending=False)
    .reset_index(drop=True)
)

In [52]:
stroke_centers_by_county.head(20)

,County,Total Stroke Centers
0,Kings,12
1,Suffolk,11
2,New York,11
3,Nassau,11
4,Westchester,10
5,Queens,9
6,Bronx,6
7,Erie,4
8,Monroe,4
9,Richmond,3


In [53]:
#Finding # of each type of stroke center in each county
stroke_centers_by_county_detailed = (
    df.groupby(["County", "NYSDOH Level of Designation"])["Hospital Name"]
    .count()
    .unstack(fill_value=0)
    .reset_index()
)
stroke_centers_by_county_detailed["Total Stroke Centers"] = stroke_centers_by_county_detailed.iloc[:, 1:].sum(axis=1)
stroke_centers_by_county_detailed.columns.name = None

In [54]:
stroke_centers_by_county_detailed.head(20)

,County,Comprehensive Stroke Center,Primary Stroke Center,Thrombectomy Capable Stroke Center,Total Stroke Centers
0,Albany,1,1,0,2
1,Allegany,0,1,0,1
2,Bronx,1,4,1,6
3,Broome,1,1,0,2
4,Cattaraugus,0,1,0,1
5,Cayuga,0,1,0,1
6,Chemung,0,1,0,1
7,Columbia,0,1,0,1
8,Cortland,0,1,0,1
9,Dutchess,0,1,2,3


#### Importing and cleaning CT 

In [55]:
from io import StringIO

url = "https://portal.ct.gov/dph/emergency-medical-services/ems/certified-stroke-centers"
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}

response = requests.get(url, headers=headers)
response.raise_for_status()

tables = pd.read_html(StringIO(response.text))
print(f"{len(tables)} tables found")
for i, t in enumerate(tables):
    print(f"Table {i}: {t.shape}")

ct_df = max(tables, key=lambda t: t.shape[0])
print(ct_df.shape)
ct_df.head()

3 tables found
Table 0: (2, 3)
Table 1: (1, 2)
Table 2: (30, 4)
(30, 4)


,0,1,2,3
0,HOSPITAL,CERTIFICATION CATEGORY,DATE OF CERTIFICATION,CERTIFYING ORGANIZATION
1,Bridgeport Hospital,Thrombectomy-Capable CONTACT:Andressa Goncalves,10/26/2024 - 10/26/2026 203.384.3104,The Joint Commission andressa.goncalves2@bpth...
2,Bridgeport Hospital Milford,Acute Stroke Center CONTACT:Andressa Goncalves,1/24/2026 - 1/24/2028 203.384.3104,The Joint Commission andressa.goncalves2@bpth...
3,Charlotte Hungerford Hospital,Primary Stroke Center CONTACT: Robyn Hernandez,1/31/2024 - 1/31/2026 860.459.1775,The Joint Commission robyn.hernandez@hhchealt...
4,Danbury Hospital,Thrombectomy-Capable CONTACT: Lauren Henriques,4/27/2024 - 4/27/2026 203.739.6973,The Joint Commission lauren.henriques@nuvance...


In [56]:
ct_df = tables[2].copy()
ct_df.columns = ct_df.iloc[0]        # set first row as header
ct_df = ct_df.drop(0).reset_index(drop=True)  # drop that row

In [57]:
ct_df.head(20)

,HOSPITAL,CERTIFICATION CATEGORY,DATE OF CERTIFICATION,CERTIFYING ORGANIZATION
0,Bridgeport Hospital,Thrombectomy-Capable CONTACT:Andressa Goncalves,10/26/2024 - 10/26/2026 203.384.3104,The Joint Commission andressa.goncalves2@bpth...
1,Bridgeport Hospital Milford,Acute Stroke Center CONTACT:Andressa Goncalves,1/24/2026 - 1/24/2028 203.384.3104,The Joint Commission andressa.goncalves2@bpth...
2,Charlotte Hungerford Hospital,Primary Stroke Center CONTACT: Robyn Hernandez,1/31/2024 - 1/31/2026 860.459.1775,The Joint Commission robyn.hernandez@hhchealt...
3,Danbury Hospital,Thrombectomy-Capable CONTACT: Lauren Henriques,4/27/2024 - 4/27/2026 203.739.6973,The Joint Commission lauren.henriques@nuvance...
4,Day Kimball Healthcare,Primary Stroke Center CONTACT: John O'Keefe,8/06/2024 - 8/06/2026 860.928.6541,The Joint Commission jokeefe@daykimball.org
5,Greenwich Hospital,Primary Stroke Center CONTACT: Sheryl Feldheim,6/14/2025 - 6/14/2027 203.863.3295,The Joint Commission sheryl.feldheim@greenwic...
6,Griffin Hospital,Primary Stroke Center CONTACT: Morgan D'Amore,12/04/2024 - 12/04/2026 203.732.7587,The Joint Commission mdamore@griffinhealth.org
7,Hartford Hospital,Comprehensive Stroke Center CONTACT: Jennie N...,3/12/2026 - 3/12/2028 860.972.1940,The Joint Commission jennie.nazario@hhchealth...
8,THOCC - New Britain General,Primary Stroke Center CONTACT: Kristen Hickey,7/26/2025 - 7/26/2027 203.910.6455,The Joint Commission kristen.hickey@hhchealth...
9,THOCC - Bradley General,Primary Stroke Center CONTACT: Kristen Hickey,7/26/2025 - 7/26/2027 203.910.6455,The Joint Commission kristen.hickey@hhchealth...


In [58]:
ct_df = ct_df.drop(columns=['DATE OF CERTIFICATION', 'CERTIFYING ORGANIZATION'])

In [59]:
ct_df.head()

,HOSPITAL,CERTIFICATION CATEGORY
0,Bridgeport Hospital,Thrombectomy-Capable CONTACT:Andressa Goncalves
1,Bridgeport Hospital Milford,Acute Stroke Center CONTACT:Andressa Goncalves
2,Charlotte Hungerford Hospital,Primary Stroke Center CONTACT: Robyn Hernandez
3,Danbury Hospital,Thrombectomy-Capable CONTACT: Lauren Henriques
4,Day Kimball Healthcare,Primary Stroke Center CONTACT: John O'Keefe


In [60]:
ct_df.iloc[:, 1] = ct_df.iloc[:, 1].str.split("CONTACT:").str[0].str.strip()

In [61]:
ct_df.head()

,HOSPITAL,CERTIFICATION CATEGORY
0,Bridgeport Hospital,Thrombectomy-Capable
1,Bridgeport Hospital Milford,Acute Stroke Center
2,Charlotte Hungerford Hospital,Primary Stroke Center
3,Danbury Hospital,Thrombectomy-Capable
4,Day Kimball Healthcare,Primary Stroke Center


In [62]:
ct_latlong = pd.read_csv('ct_stroke_centers_geocoded.csv')
ct_latlong.head()

,name,group,latitude,longitude
0,Bridgeport Hospital Milford Campus,Basic,41.216545,-73.065360
1,Charlotte Hungerford Hospital,Basic,41.792271,-73.133769
2,Day Kimball Hospital,Basic,41.906093,-71.913028
3,Greenwich Hospital,Basic,41.034299,-73.630646
4,Hospital of Central Connecticut,Basic,41.661381,-72.786712


In [64]:
len(ct_df)

29

In [66]:
!conda install -c conda-forge geopandas -y

Jupyter detected...
2 channel Terms of Service accepted
Retrieving notices: done
Channels:
 - conda-forge
 - defaults
Platform: osx-arm64
Solving environment: done


==> WARNING: A newer version of conda exists. <==
    current version: 25.11.1
    latest version: 26.5.3

Please update conda by running

    $ conda update -n base -c defaults conda



## Package Plan ##

  environment location: /opt/anaconda3

  added / updated specs:
    - geopandas


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    ca-certificates-2026.6.17  |       hbd8a1cb_0         126 KB  conda-forge
    certifi-2026.6.17          |     pyhd8ed1ab_0         131 KB  conda-forge
    conda-26.1.1               |  py313h8f79df9_0         1.2 MB  conda-forge
    freexl-2.0.0               |       ha3de405_0          52 KB
    geopandas-1.1.4            |     pyhd8ed1ab_0           8 KB  conda-forge
    geopandas-base-1.1.

In [85]:
import requests

def get_county(lat, lon, vintage=419):
    url = "https://geocoding.geo.census.gov/geocoder/geographies/coordinates"
    params = {
        "x": lon,
        "y": lat,
        "benchmark": "Public_AR_Current",
        "vintage": vintage,
        "format": "json"
    }
    r = requests.get(url, params=params)
    result = r.json()
    try:
        county_info = result["result"]["geographies"]["Counties"][0]
        return county_info["NAME"], county_info["GEOID"]
    except (KeyError, IndexError):
        return None, None

ct_latlong[["County", "County_FIPS"]] = ct_latlong.apply(
    lambda row: pd.Series(get_county(row["latitude"], row["longitude"])),
    axis=1
)

In [86]:
ct_df_with_county.columns

Index(['name', 'group', 'County', 'FIPS'], dtype='object')

In [87]:
ct_df_with_county.head()

,name,group,County,FIPS
0,Bridgeport Hospital Milford Campus,Basic,South Central Connecticut,09170
1,Charlotte Hungerford Hospital,Basic,Northwest Hills,09160
2,Day Kimball Hospital,Basic,Northeastern Connecticut,09150
3,Greenwich Hospital,Basic,Western Connecticut,09190
4,Hospital of Central Connecticut,Basic,Capitol,09110
